In [0]:
%pip install lightgbm scikit-learn pandas numpy mlflow shap

# Restart Python after installing (run this cell alone, then continue)
dbutils.library.restartPython()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 136.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 109.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 606.1/606.1 kB 18.9 MB/s eta 0:00:00
  Attempting uninstall: blinker
    Found existing installation: blinker 1.7.0
    Not uninstalling blinker at /usr/lib/python3/dist-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-7e679f3e-efb1-433f-a0f6-398ff53ac761
    Can't uninstall 'blinker'. No files were found to

**Imports**

 These are all the tools we need. Think of imports like opening toolboxes.


In [0]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.lightgbm
import lightgbm as lgb
import shap

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings("ignore")

print("All imports successful.")


All imports successful.


 **Load the data**

IMPORTANT: Upload application_train.csv to Databricks first.
 Go to: Data > Add Data > Upload File > choose application_train.csv
 Then copy the file path shown and paste it below.
 The path will look like: /FileStore/tables/application_train.csv


In [0]:
# Change this path to wherever you uploaded the file in Databricks
DATA_PATH = "/Volumes/workspace/default/hackbricks/application_train.csv"

df = pd.read_csv(DATA_PATH)

print(f"Dataset loaded. Shape: {df.shape}")
print(f"Total loan applications: {len(df):,}")
print(f"Default rate (TARGET=1): {df['TARGET'].mean():.2%}")
# TARGET=1 means the person defaulted (didn't repay). TARGET=0 means repaid.

Dataset loaded. Shape: (307511, 122)
Total loan applications: 307,511
Default rate (TARGET=1): 8.07%


**Quick data exploration**

 Let's understand what we're working with before touching anything.


In [0]:
print("=== First 5 rows ===")
display(df.head())

print("\n=== Target distribution ===")
print(df['TARGET'].value_counts())
# You'll see ~8% are defaults (TARGET=1). This is an imbalanced dataset.

print("\n=== Missing values (top 20 columns) ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
print(missing_df[missing_df['missing_count'] > 0].sort_values('missing_pct', ascending=False).head(20))

=== First 5 rows ===


SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.018801,-9461,-637,-3648.0,-2120,null,1,1,0,1,1,0,Laborers,1.0,2,2,WEDNESDAY,10,0,0,0,0,0,0,Business Entity Type 3,0.0830369673913225,0.2629485927471776,0.1393757800997895,0.0247,0.0369,0.9722,0.6192,0.0143,0.0,0.069,0.0833,0.125,0.0369,0.0202,0.019,0.0,0.0,0.0252,0.0383,0.9722,0.6341,0.0144,0.0,0.069,0.0833,0.125,0.0377,0.022,0.0198,0.0,0.0,0.025,0.0369,0.9722,0.6243,0.0144,0.0,0.069,0.0833,0.125,0.0375,0.0205,0.0193,0.0,0.0,reg oper account,block of flats,0.0149,"Stone, brick",No,2.0,2.0,2.0,2.0,-1134.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,Family,State servant,Higher education,Married,House / apartment,0.0035409999999999,-16765,-1188,-1186.0,-291,null,1,1,0,1,1,0,Core staff,2.0,1,1,MONDAY,11,0,0,0,0,0,0,School,0.3112673113812225,0.6222457752555098,null,0.0959,0.0529,0.9851,0.7959999999999999,0.0605,0.08,0.0345,0.2917,0.3333,0.013,0.0773,0.0549,0.0039,0.0098,0.0924,0.0538,0.9851,0.804,0.0497,0.0806,0.0345,0.2917,0.3333,0.0128,0.079,0.0554,0.0,0.0,0.0968,0.0529,0.9851,0.7987,0.0608,0.08,0.0345,0.2917,0.3333,0.0132,0.0787,0.0558,0.0039,0.01,reg oper account,block of flats,0.0714,Block,No,1.0,0.0,1.0,0.0,-828.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,135000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.010032,-19046,-225,-4260.0,-2531,26.0,1,1,1,1,1,0,Laborers,1.0,2,2,MONDAY,9,0,0,0,0,0,0,Government,null,0.5559120833904428,0.7295666907060153,null,null,null,null,null,null,null,null,null,null,null,null,


=== Target distribution ===
TARGET
0    282686
1     24825
Name: count, dtype: int64

=== Missing values (top 20 columns) ===
                          missing_count  missing_pct
COMMONAREA_MEDI                  214865        69.87
COMMONAREA_AVG                   214865        69.87
COMMONAREA_MODE                  214865        69.87
NONLIVINGAPARTMENTS_MEDI         213514        69.43
NONLIVINGAPARTMENTS_MODE         213514        69.43
NONLIVINGAPARTMENTS_AVG          213514        69.43
FONDKAPREMONT_MODE               210295        68.39
LIVINGAPARTMENTS_MODE            210199        68.35
LIVINGAPARTMENTS_MEDI            210199        68.35
LIVINGAPARTMENTS_AVG             210199        68.35
FLOORSMIN_MODE                   208642        67.85
FLOORSMIN_MEDI                   208642        67.85
FLOORSMIN_AVG                    208642        67.85
YEARS_BUILD_MODE                 204488        66.50
YEARS_BUILD_MEDI                 204488        66.50
YEARS_BUILD_AVG          

**Data Cleaning & Feature Engineering**

 We need to:
 1. Drop columns with too many missing values (>60% missing is useless)
 2. Convert text columns (categorical) to numbers — ML needs numbers
 3. Fill remaining missing values with the median


In [0]:
# Step 1: Drop columns where more than 60% of values are missing
threshold = 0.6
cols_to_drop = [col for col in df.columns if df[col].isnull().mean() > threshold]
df = df.drop(columns=cols_to_drop)
print(f"Dropped {len(cols_to_drop)} columns with >60% missing values.")
print(f"Remaining columns: {df.shape[1]}")

# Step 2: Separate our target (what we want to predict) from features (inputs)
TARGET_COL = "TARGET"
ID_COL = "SK_ID_CURR"  # This is just the loan application ID, not a useful feature

y = df[TARGET_COL]                          # What we want to predict (0 or 1)
X = df.drop(columns=[TARGET_COL, ID_COL])  # Everything else = inputs to the model

# Step 3: Find text (categorical) columns — LightGBM needs numbers
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
print(f"\nCategorical columns to encode: {categorical_cols}")

# Convert each text column to numbers using LabelEncoder
# Example: "Male"→1, "Female"→0
le = LabelEncoder()
for col in categorical_cols:
    X[col] = X[col].fillna("MISSING")     # Fill missing text with "MISSING"
    X[col] = le.fit_transform(X[col].astype(str))

# Step 4: Fill remaining missing numbers with column median
X = X.fillna(X.median())

print(f"\nFinal feature matrix shape: {X.shape}")
print("Data cleaning complete.")


Dropped 17 columns with >60% missing values.
Remaining columns: 105

Categorical columns to encode: ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']

Final feature matrix shape: (307511, 103)
Data cleaning complete.


**Train/Validation/Test Split**

 We split our data into 3 parts:
 - Train (70%): Model learns from this
 - Validation (15%): We tune/monitor the model on this during training
 - Test (15%): Final evaluation — model never sees this during training
               This is what we use for Champion vs Challenger comparison


In [0]:
# First split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,        # random_state=42 means same split every time you run
    stratify=y              # stratify=y ensures both splits have same % of defaults
)

# Second split: split the 30% into 15% validation + 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print(f"Train size:      {len(X_train):,} rows ({len(X_train)/len(X):.0%})")
print(f"Validation size: {len(X_val):,} rows ({len(X_val)/len(X):.0%})")
print(f"Test size:       {len(X_test):,} rows ({len(X_test)/len(X):.0%})")
print(f"\nDefault rate in train:      {y_train.mean():.2%}")
print(f"Default rate in validation: {y_val.mean():.2%}")
print(f"Default rate in test:       {y_test.mean():.2%}")
# These % should all be very similar — that's what stratify=y ensures

Train size:      215,257 rows (70%)
Validation size: 46,127 rows (15%)
Test size:       46,127 rows (15%)

Default rate in train:      8.07%
Default rate in validation: 8.07%
Default rate in test:       8.07%


**Define Model Parameters**

 LightGBM (LGBM) is a gradient boosting model — very popular for tabular data
 like loan applications. It's fast, handles missing values, and wins Kaggle.

 Key parameters explained:
 - n_estimators: how many decision trees to build (more = potentially better but slower)
 - learning_rate: how much each tree corrects the previous (lower = more careful)
 - num_leaves: complexity of each tree (higher = more complex patterns learned)
 - scale_pos_weight: since only 8% are defaults, we tell the model to pay more
                     attention to defaults (penalize missing them more)


In [0]:
# Calculate class imbalance ratio for scale_pos_weight
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count  # ~11 (there are 11x more non-defaults)
print(f"Class imbalance ratio (scale_pos_weight): {scale_pos_weight:.2f}")

MODEL_PARAMS = {
    "n_estimators": 500,
    "learning_rate": 0.05,
    "num_leaves": 63,
    "max_depth": -1,          # -1 means no limit
    "min_child_samples": 50,
    "subsample": 0.8,         # Use 80% of rows per tree (prevents overfitting)
    "colsample_bytree": 0.8,  # Use 80% of features per tree
    "scale_pos_weight": scale_pos_weight,
    "random_state": 42,
    "n_jobs": -1,             # Use all CPU cores
    "verbose": -1             # Suppress LightGBM's own output
}

print("\nModel parameters set.")


Class imbalance ratio (scale_pos_weight): 11.39

Model parameters set.


**Train Model with MLflow Tracking**

This is the most important cell. We wrap training in an MLflow "run"
so every parameter, metric, and artifact is automatically saved.

Think of mlflow.start_run() as opening a diary entry for this experiment.
Everything inside the 'with' block gets logged to that diary entry.


In [0]:
# Set the MLflow experiment name — this groups all your runs together
# You'll see this experiment in the MLflow UI in Databricks
EXPERIMENT_NAME = "/Users/rajath2010rrp@gmail.com/hackbricks_credit_risk"
# NOTE: Replace 'your_email' with your actual Databricks username/email

mlflow.set_experiment(EXPERIMENT_NAME)

print("Starting MLflow training run...")
print("This will take 2-5 minutes depending on cluster size.\n")

with mlflow.start_run(run_name="baseline_lgbm_champion") as run:

    # Save the run ID — we'll need this later to register the model
    RUN_ID = run.info.run_id
    print(f"MLflow Run ID: {RUN_ID}")

    # --- Log all hyperparameters ---
    # This saves every parameter so you can reproduce this exact model later
    mlflow.log_params(MODEL_PARAMS)
    mlflow.log_param("train_size", len(X_train))
    mlflow.log_param("val_size", len(X_val))
    mlflow.log_param("test_size", len(X_test))
    mlflow.log_param("n_features", X_train.shape[1])
    mlflow.log_param("target_col", TARGET_COL)
    mlflow.log_param("model_type", "LightGBM")

    # --- Train the model ---
    model = lgb.LGBMClassifier(**MODEL_PARAMS)

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],       # Monitor performance on validation set
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=True),
            # Stop early if validation AUC doesn't improve for 50 rounds
            # This prevents overfitting
            lgb.log_evaluation(period=50) # Print progress every 50 rounds
        ]
    )

    # --- Evaluate on all 3 splits ---
    train_preds = model.predict_proba(X_train)[:, 1]  # Probability of default
    val_preds   = model.predict_proba(X_val)[:, 1]
    test_preds  = model.predict_proba(X_test)[:, 1]

    train_auc = roc_auc_score(y_train, train_preds)
    val_auc   = roc_auc_score(y_val,   val_preds)
    test_auc  = roc_auc_score(y_test,  test_preds)

    print(f"\n=== Model Performance ===")
    print(f"Train AUC:      {train_auc:.4f}")
    print(f"Validation AUC: {val_auc:.4f}")
    print(f"Test AUC:       {test_auc:.4f}")
    # AUC of 0.70+ is good. 0.75+ is excellent for credit risk.
    # If train AUC >> val AUC, the model is overfitting.

    # --- Log all metrics to MLflow ---
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    mlflow.log_metric("test_auc", test_auc)
    mlflow.log_metric("best_iteration", model.best_iteration_)

    # Gini coefficient = 2 * AUC - 1 (used in banking/credit industry)
    # A Gini of 0.5 = very good for credit scoring
    gini = 2 * test_auc - 1
    mlflow.log_metric("gini_coefficient", gini)
    print(f"Gini Coefficient: {gini:.4f}")

    # --- Log the model itself ---
    # This saves the trained model file so you can load it later
    # --- Log the model itself ---
    from mlflow.models.signature import infer_signature

    # infer_signature tells MLflow:
    # - what the INPUT looks like (X_train = feature columns, their names and data types)
    # - what the OUTPUT looks like (train_preds = the probability scores the model returns)
    # Unity Catalog REQUIRES this — it won't accept a model without it
    signature = infer_signature(X_train, train_preds)

    mlflow.lightgbm.log_model(
        model,
        artifact_path="model",
        registered_model_name="credit-risk-champion",
        signature=signature,
        input_example=X_train.head(5)  # saves 5 sample rows so MLflow can show an example
    )

    # --- Log feature importances ---
    # Which features matter most for predictions?
    feature_importance_df = pd.DataFrame({
        'feature': X_train.columns,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)

    # Save as a CSV artifact in MLflow
    feature_importance_df.to_csv("/tmp/feature_importances.csv", index=False)
    mlflow.log_artifact("/tmp/feature_importances.csv")

    print(f"\n=== Top 10 Most Important Features ===")
    print(feature_importance_df.head(10).to_string(index=False))

    # --- Compute SHAP values (for LLM grounding later) ---
    # --- Compute SHAP values (for LLM grounding later) ---
    print("\nComputing SHAP values (this takes ~1 min)...")
    explainer = shap.TreeExplainer(model)

    # Only compute SHAP for a sample (full dataset would take too long)
    X_shap_sample = X_test.sample(n=500, random_state=42)
    shap_values = explainer.shap_values(X_shap_sample)

    # Newer versions of SHAP return a 3D array (n_samples, n_features, n_classes)
    # Older versions return a list [class_0_array, class_1_array]
    # This handles both cases safely
    if isinstance(shap_values, list):
        # Old SHAP format: list of arrays, index [1] = class 1 (default)
        shap_array = shap_values[1]
    elif shap_values.ndim == 3:
        # New SHAP format: 3D array, [:, :, 1] = class 1 (default)
        shap_array = shap_values[:, :, 1]
    else:
        # Already 2D (binary output directly)
        shap_array = shap_values

    shap_df = pd.DataFrame(shap_array, columns=X_shap_sample.columns)
    shap_df.to_csv("/tmp/shap_values_sample.csv", index=False)
    mlflow.log_artifact("/tmp/shap_values_sample.csv")

    print("SHAP values computed and logged.")
    print(f"SHAP values shape: {shap_array.shape}  →  (samples, features)")
    print(f"\nMLflow run completed. Run ID: {RUN_ID}")

print("\nTraining complete!")

Starting MLflow training run...
This will take 2-5 minutes depending on cluster size.

MLflow Run ID: fec526314a264c649e3df338c806122c
Training until validation scores don't improve for 50 rounds
[50]	valid_0's binary_logloss: 0.534476
Early stopping, best iteration is:
[1]	valid_0's binary_logloss: 0.275985

=== Model Performance ===
Train AUC:      0.7298
Validation AUC: 0.7184
Test AUC:       0.7231
Gini Coefficient: 0.4463


2026/04/11 13:19:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-3f31c628-f679.cloud.databricks.com/ml/experiments/2608798312767667/models/m-b125fb6e17044071b02e6dfd812d6e98?o=7474656095211125
Registered model 'credit-risk-champion' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]

🔗 Created version '2' of model 'workspace.default.credit-risk-champion': https://dbc-3f31c628-f679.cloud.databricks.com/explore/data/models/workspace/default/credit-risk-champion/version/2?o=7474656095211125



=== Top 10 Most Important Features ===
               feature  importance
          EXT_SOURCE_1          17
          EXT_SOURCE_2          12
          EXT_SOURCE_3           7
           CODE_GENDER           6
   NAME_EDUCATION_TYPE           5
            DAYS_BIRTH           5
         DAYS_EMPLOYED           3
            AMT_CREDIT           2
REG_CITY_NOT_WORK_CITY           1
       OCCUPATION_TYPE           1

Computing SHAP values (this takes ~1 min)...
SHAP values computed and logged.
SHAP values shape: (500, 103)  →  (samples, features)

MLflow run completed. Run ID: fec526314a264c649e3df338c806122c

Training complete!


 **Set Champion alias in Model Registry**
 
MLflow Model Registry stores all versions of your model.
We now set the LATEST version as the official "Champion".

In [0]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Get all versions of the model and find the latest one
# Unity Catalog requires fully qualified name: catalog.schema.model
model_name = "workspace.default.credit-risk-champion"
all_versions = client.search_model_versions(f"name='{model_name}'")

# Sort by version number (descending) and get the latest
latest_version = max([int(v.version) for v in all_versions])

print(f"Registering model version {latest_version} as Champion...")

# Set alias "Champion" on this version
client.set_registered_model_alias(
    name=model_name,
    alias="Champion",
    version=str(latest_version)
)

# Add description so evaluators know what this is
client.update_model_version(
    name=model_name,
    version=str(latest_version),
    description=f"Baseline LightGBM trained on Home Credit data. Test AUC: {test_auc:.4f}, Gini: {gini:.4f}. Run ID: {RUN_ID}"
)

print(f"Champion model registered successfully!")
print(f"Model: {model_name}, Version: {latest_version}, Alias: Champion")

Registering model version 2 as Champion...
Champion model registered successfully!
Model: workspace.default.credit-risk-champion, Version: 2, Alias: Champion


**Save test set for Champion vs Challenger comparison later**

We save X_test and y_test so Notebook 3 can use the EXACT same test set
when comparing Champion vs Challenger. This ensures a fair comparison.

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Convert pandas DataFrames to Spark and save as Delta tables
test_df = X_test.copy()
test_df['TARGET'] = y_test.values
test_df['SK_ID_CURR'] = range(len(test_df))  # Add a simple ID column

spark_test_df = spark.createDataFrame(test_df)
spark_test_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.champion_test_set")
print("Test set saved to Delta table: hackbricks.champion_test_set")
print(f"Rows saved: {len(test_df):,}")

# Also save the feature column names — we'll need these for drift detection
feature_cols = list(X_train.columns)
feature_cols_df = pd.DataFrame({'feature_name': feature_cols})
spark.createDataFrame(feature_cols_df).write.format("delta").mode("overwrite").saveAsTable("workspace.default.feature_columns")

print("Feature column list saved to: hackbricks.feature_columns")

Test set saved to Delta table: hackbricks.champion_test_set
Rows saved: 46,127
Feature column list saved to: hackbricks.feature_columns
